# Prompts and Structured Output

## What you'll learn
- How a **system prompt** sets the model's role and rules
- How to give the model a small table of data to reason about
- How to ask for **structured output** (JSON that matches a schema) instead of free text
- How to turn that JSON straight into a pandas DataFrame

In [ ]:
import json
import pandas as pd

df = pd.read_csv("data/fpa_actuals_budget.csv")

# Derived lines every FP&A pack uses (all amounts in USD thousands).
df["Gross_Profit"] = df["Revenue"] - df["COGS"]
df["Budget_Gross_Profit"] = df["Budget_Revenue"] - df["Budget_COGS"]
df["Operating_Income"] = df["Gross_Profit"] - df["OpEx"]
df["Budget_Operating_Income"] = df["Budget_Gross_Profit"] - df["Budget_OpEx"]

print(df.shape)
df.head()

In [ ]:
import os
import importlib
from getpass import getpass

# Choose your provider: "anthropic" (Claude) or "openai" (ChatGPT).
PROVIDER = "anthropic"

MODELS = {"anthropic": "claude-opus-5", "openai": "gpt-5.6"}
KEY_NAMES = {"anthropic": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY"}
MODEL = MODELS[PROVIDER]
sdk = importlib.import_module(PROVIDER)   # the anthropic or openai package


def make_client():
    """Create an API client, or return None (with a message) if there is no key."""
    key_name = KEY_NAMES[PROVIDER]
    if not os.environ.get(key_name):
        print(f"{key_name} is not set. Paste your key below, or press Enter to skip.")
        print("(The key stays in memory only. See 02_api_key_and_first_call.ipynb.)")
        try:
            key = getpass(f"{key_name}: ").strip()
        except Exception:
            key = ""
        if not key:
            print("No key, so the model cells in this notebook will be skipped.")
            return None
        os.environ[key_name] = key
    if PROVIDER == "anthropic":
        return sdk.Anthropic()
    return sdk.OpenAI()


client = make_client()
print("Provider:", PROVIDER, "| Model:", MODEL, "| Ready:", client is not None)

In [ ]:
def explain_error(error):
    """Turn an API error into a plain message."""
    if isinstance(error, sdk.AuthenticationError):
        return "The API key was rejected. Check that it is correct and still active."
    if isinstance(error, sdk.RateLimitError):
        return "Rate limit or usage quota reached. Wait a minute, or check your account billing."
    if isinstance(error, sdk.APIConnectionError):
        return "Could not reach the API. Check your internet connection or company proxy."
    if isinstance(error, sdk.APIStatusError):
        return f"The API returned an error ({error.status_code}): {error.message}"
    return f"Unexpected API error: {error}"

In [ ]:
def ask_llm(prompt, system=None):
    """Send one prompt to the chosen provider and return the reply text."""
    if client is None:
        return "(Skipped: no API key. See 02_api_key_and_first_call.ipynb.)"
    try:
        if PROVIDER == "anthropic":
            extra = {"system": system} if system else {}
            response = client.messages.create(
                model=MODEL,
                max_tokens=16000,
                messages=[{"role": "user", "content": prompt}],
                **extra,
            )
            if response.stop_reason == "refusal":
                print("The model declined this request.")
                return None
            # The reply is a list of blocks; keep the text ones.
            return "".join(block.text for block in response.content if block.type == "text")
        else:
            extra = {"instructions": system} if system else {}
            response = client.responses.create(model=MODEL, input=prompt, **extra)
            return response.output_text
    except sdk.APIError as error:
        print(explain_error(error))
        return None

## System prompts

The **system prompt** is a standing instruction that applies to the whole conversation:
who the model is, what it must and must not do, and how to format answers. The user
prompt is the actual question. Good system prompts are specific about units, tone, and
limits.

In [ ]:
SYSTEM = """You are a concise FP&A analyst. Amounts are in USD thousands.
Only use the numbers you are given. Answer in at most three bullet points."""

q2 = df[(df["Month"] >= "2025-04") & (df["Month"] <= "2025-06")]
table = q2.groupby("Region")[["Revenue", "Budget_Revenue"]].sum().round(1)
table["variance"] = table["Revenue"] - table["Budget_Revenue"]

prompt = f"""Here is Q2 2025 revenue by region:

{table.to_string()}

Which region needs attention, and what would you check next?"""

reply = ask_llm(prompt, system=SYSTEM)
if reply:
    print(reply)

**Why a small table?** The model reads everything you send, and you pay for it. Summarise
with pandas first and send only the result. The raw CSV has hundreds of rows; the model
needs four.

## Structured output

Free text is fine for people, but code needs predictable fields. Both providers can
**guarantee** a reply that matches a JSON schema:

- **Anthropic:** pass `output_config={"format": {"type": "json_schema", "schema": ...}}`.
- **OpenAI:** pass `text={"format": {"type": "json_schema", "name": ..., "strict": True, "schema": ...}}`.

A schema lists the fields, their types, and which are required. Setting
`"additionalProperties": False` and listing every field in `required` keeps the output
strict for both providers.

In [ ]:
def ask_json(prompt, schema, system=None):
    """Ask for a reply that matches a JSON schema and return it as a Python dict."""
    if client is None:
        return None
    try:
        if PROVIDER == "anthropic":
            extra = {"system": system} if system else {}
            response = client.messages.create(
                model=MODEL,
                max_tokens=16000,
                messages=[{"role": "user", "content": prompt}],
                output_config={"format": {"type": "json_schema", "schema": schema}},
                **extra,
            )
            text = "".join(block.text for block in response.content if block.type == "text")
        else:
            extra = {"instructions": system} if system else {}
            response = client.responses.create(
                model=MODEL,
                input=prompt,
                text={"format": {"type": "json_schema", "name": "answer",
                                 "strict": True, "schema": schema}},
                **extra,
            )
            text = response.output_text
        return json.loads(text)
    except sdk.APIError as error:
        print(explain_error(error))
    except json.JSONDecodeError:
        print("The reply was not valid JSON:", text[:200])
    return None

In [ ]:
COMMENTARY_SCHEMA = {
    "type": "object",
    "properties": {
        "headline": {"type": "string", "description": "One-sentence summary"},
        "regions": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "region": {"type": "string"},
                    "status": {"type": "string", "enum": ["favourable", "unfavourable", "on budget"]},
                    "comment": {"type": "string"},
                },
                "required": ["region", "status", "comment"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["headline", "regions"],
    "additionalProperties": False,
}

result = ask_json(prompt, COMMENTARY_SCHEMA, system=SYSTEM)
if result:
    print(result["headline"])
    display(pd.DataFrame(result["regions"]))

Because the reply is a dict, you can check it, store it, or load it into a DataFrame
without any text parsing. This is how model output becomes part of a real pipeline.

## Practice

1. Change `SYSTEM` so the answer is written for a non-finance audience. Compare the replies.
2. Send the Q2 2025 **OpEx** variance by product line and ask which line to investigate.
3. Write a schema with fields `product_line` (string), `risk` (`"low"`, `"medium"`, `"high"`)
   and `reason` (string), and use `ask_json` to rate each product line's Q2 2025 revenue risk.
   (Hint: wrap the list in an object, as `COMMENTARY_SCHEMA` does.)
4. Load the result of exercise 3 into a DataFrame and sort it by risk.

In [ ]:
# Your practice code here

## Summary

- The system prompt sets role, rules, units, and format for the whole conversation.
- Summarise with pandas and send the model a small table, not the raw data.
- Structured output returns JSON that matches your schema, ready for code.
- Anthropic uses `output_config`; OpenAI uses `text.format`; the schema itself is the same.

## Practice Solutions

In [ ]:
import json
import pandas as pd

df = pd.read_csv("data/fpa_actuals_budget.csv")

# Derived lines every FP&A pack uses (all amounts in USD thousands).
df["Gross_Profit"] = df["Revenue"] - df["COGS"]
df["Budget_Gross_Profit"] = df["Budget_Revenue"] - df["Budget_COGS"]
df["Operating_Income"] = df["Gross_Profit"] - df["OpEx"]
df["Budget_Operating_Income"] = df["Budget_Gross_Profit"] - df["Budget_OpEx"]




import os
import importlib
from getpass import getpass

# Choose your provider: "anthropic" (Claude) or "openai" (ChatGPT).
PROVIDER = "anthropic"

MODELS = {"anthropic": "claude-opus-5", "openai": "gpt-5.6"}
KEY_NAMES = {"anthropic": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY"}
MODEL = MODELS[PROVIDER]
sdk = importlib.import_module(PROVIDER)   # the anthropic or openai package


def make_client():
    """Create an API client, or return None (with a message) if there is no key."""
    key_name = KEY_NAMES[PROVIDER]
    if not os.environ.get(key_name):
        print(f"{key_name} is not set. Paste your key below, or press Enter to skip.")
        print("(The key stays in memory only. See 02_api_key_and_first_call.ipynb.)")
        try:
            key = getpass(f"{key_name}: ").strip()
        except Exception:
            key = ""
        if not key:
            print("No key, so the model cells in this notebook will be skipped.")
            return None
        os.environ[key_name] = key
    if PROVIDER == "anthropic":
        return sdk.Anthropic()
    return sdk.OpenAI()


client = make_client()
print("Provider:", PROVIDER, "| Model:", MODEL, "| Ready:", client is not None)



def explain_error(error):
    """Turn an API error into a plain message."""
    if isinstance(error, sdk.AuthenticationError):
        return "The API key was rejected. Check that it is correct and still active."
    if isinstance(error, sdk.RateLimitError):
        return "Rate limit or usage quota reached. Wait a minute, or check your account billing."
    if isinstance(error, sdk.APIConnectionError):
        return "Could not reach the API. Check your internet connection or company proxy."
    if isinstance(error, sdk.APIStatusError):
        return f"The API returned an error ({error.status_code}): {error.message}"
    return f"Unexpected API error: {error}"



def ask_llm(prompt, system=None):
    """Send one prompt to the chosen provider and return the reply text."""
    if client is None:
        return "(Skipped: no API key. See 02_api_key_and_first_call.ipynb.)"
    try:
        if PROVIDER == "anthropic":
            extra = {"system": system} if system else {}
            response = client.messages.create(
                model=MODEL,
                max_tokens=16000,
                messages=[{"role": "user", "content": prompt}],
                **extra,
            )
            if response.stop_reason == "refusal":
                print("The model declined this request.")
                return None
            # The reply is a list of blocks; keep the text ones.
            return "".join(block.text for block in response.content if block.type == "text")
        else:
            extra = {"instructions": system} if system else {}
            response = client.responses.create(model=MODEL, input=prompt, **extra)
            return response.output_text
    except sdk.APIError as error:
        print(explain_error(error))
        return None



def ask_json(prompt, schema, system=None):
    """Ask for a reply that matches a JSON schema and return it as a Python dict."""
    if client is None:
        return None
    try:
        if PROVIDER == "anthropic":
            extra = {"system": system} if system else {}
            response = client.messages.create(
                model=MODEL,
                max_tokens=16000,
                messages=[{"role": "user", "content": prompt}],
                output_config={"format": {"type": "json_schema", "schema": schema}},
                **extra,
            )
            text = "".join(block.text for block in response.content if block.type == "text")
        else:
            extra = {"instructions": system} if system else {}
            response = client.responses.create(
                model=MODEL,
                input=prompt,
                text={"format": {"type": "json_schema", "name": "answer",
                                 "strict": True, "schema": schema}},
                **extra,
            )
            text = response.output_text
        return json.loads(text)
    except sdk.APIError as error:
        print(explain_error(error))
    except json.JSONDecodeError:
        print("The reply was not valid JSON:", text[:200])
    return None


q2 = df[(df["Month"] >= "2025-04") & (df["Month"] <= "2025-06")]

# 1.
plain = "You explain finance to people with no finance background. Use plain words, no jargon."
table = q2.groupby("Region")[["Revenue", "Budget_Revenue"]].sum().round(1)
print(ask_llm(f"Q2 2025 revenue by region (USD thousands):\n{table.to_string()}\n"
              "Which region needs attention?", system=plain))

# 2.
opex = q2.groupby("ProductLine")[["OpEx", "Budget_OpEx"]].sum().round(1)
opex["variance"] = opex["OpEx"] - opex["Budget_OpEx"]
print(ask_llm(f"Q2 2025 OpEx by product line (USD thousands):\n{opex.to_string()}\n"
              "Which line should we investigate first, and why?"))

# 3.
RISK_SCHEMA = {
    "type": "object",
    "properties": {
        "ratings": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "product_line": {"type": "string"},
                    "risk": {"type": "string", "enum": ["low", "medium", "high"]},
                    "reason": {"type": "string"},
                },
                "required": ["product_line", "risk", "reason"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["ratings"],
    "additionalProperties": False,
}
revenue = q2.groupby("ProductLine")[["Revenue", "Budget_Revenue"]].sum().round(1)
result = ask_json(f"Rate the revenue risk of each product line:\n{revenue.to_string()}", RISK_SCHEMA)

# 4.
if result:
    ratings = pd.DataFrame(result["ratings"])
    ratings["order"] = ratings["risk"].map({"high": 0, "medium": 1, "low": 2})
    print(ratings.sort_values("order").drop(columns="order"))